# 05 — Errors, Files & I/O

**What you'll learn:**

- Why Java has exceptions and where they fit
- The `Throwable` hierarchy: `Error`, `Exception`, `RuntimeException`
- **Checked vs unchecked** exceptions — the rule the compiler enforces
- `try` / `catch` / `finally` syntax and control flow
- Multi-catch and exception rethrow
- Throwing exceptions and writing your own custom exception types
- Exception chaining with `cause`
- **`try-with-resources`** for safe cleanup of any `AutoCloseable`
- The NIO `Path` type — modern filesystem paths
- The `Files` utility — reading, writing, copying, listing
- Walking a directory tree with `Files.walk` and `Files.find`
- Serialization basics — `Serializable`, `transient`, and why you usually avoid it

This is the boring-but-essential side of any real program. The Stream API and lambdas are how you express logic; exceptions and I/O are how that logic talks to the outside world without becoming a debugging nightmare when things go wrong.

## Why exceptions

Things go wrong in real programs. A file doesn't exist. The network drops. A user supplies a string where you expected a number. Java's answer is **exceptions** — values that propagate up the call stack until something catches them, abandoning the normal return path.

The alternative — returning a special sentinel like `-1` or `null` — quietly forces every caller to remember to check. Exceptions invert the default: you opt *out* of failure handling by letting them propagate. You opt *in* by catching them.

```text
   methodA  -->  methodB  -->  methodC
                                  |
                            throw new IOException
                                  |
   methodA  <--  methodB  <-------+    propagates up
      |                                until somebody catches
      | catch (IOException e) { ... }
```

## The `Throwable` hierarchy

Every exception in Java extends `Throwable`. Two big branches:

```text
                 Throwable
                /         \
            Error          Exception
              |           /         \
        (JVM-fatal)   IOException   RuntimeException
                      SQLException   |
                          ...     NullPointerException
                                  IllegalArgumentException
                                  IndexOutOfBoundsException
                                  ArithmeticException
                                       ...
```

- **`Error`** — fatal JVM-level problems. `OutOfMemoryError`, `StackOverflowError`. Don't catch these; you can't recover from them.
- **`Exception`** (excluding `RuntimeException`) — *checked* exceptions. The compiler forces you to declare or handle them.
- **`RuntimeException`** — *unchecked*. Bugs, mostly. `NullPointerException`, `IllegalArgumentException`, `IndexOutOfBoundsException`.

## Checked vs unchecked — the rule

Java is one of the few languages with **checked exceptions**, a compile-time contract that says "this method might throw `IOException` and you must either catch it or declare that you propagate it".

| Kind | Base class | Compiler enforces? | Example |
|---|---|---|---|
| **Checked** | `Exception` (not `RuntimeException`) | Yes — must catch or `throws` | `IOException`, `SQLException` |
| **Unchecked** | `RuntimeException` | No | `NullPointerException`, `IllegalArgumentException` |
| **Errors** | `Error` | No | `OutOfMemoryError` |

The rule of thumb the language designers gave: use **checked** for recoverable conditions a caller might reasonably handle (a file might not exist; a network call might fail), and **unchecked** for programmer mistakes (you passed a negative size; you dereferenced a null). In practice, modern Java code leans unchecked — checked exceptions don't compose well with lambdas and streams.

In [ ]:
import java.io.*;

// a method that throws a CHECKED exception must declare it
public static String firstLine(String path) throws IOException {
    try (var reader = new BufferedReader(new FileReader(path))) {
        return reader.readLine();
    }
}

// callers must either catch it or declare it themselves
public static void main(String[] args) {
    try {
        System.out.println(firstLine("data.txt"));
    } catch (IOException e) {
        System.err.println("could not read: " + e.getMessage());
    }
}

// unchecked exceptions need no declaration
public static int divide(int a, int b) {
    if (b == 0) throw new IllegalArgumentException("divisor cannot be zero");
    return a / b;
}

## `try` / `catch` / `finally`

The `try` block wraps the code that might throw. Each `catch` block handles a specific exception type. The optional `finally` block runs no matter what — exception or not, return or not. It's the classic place to release a resource.

In [ ]:
try {
    var n = Integer.parseInt("not a number");      // throws NumberFormatException
    System.out.println(n);
} catch (NumberFormatException e) {
    System.err.println("bad input: " + e.getMessage());
} catch (Exception e) {
    System.err.println("unexpected: " + e);
} finally {
    System.out.println("always runs");
}

// multi-catch — handle several types the same way
try {
    risky();
} catch (IOException | SQLException e) {
    System.err.println("I/O or DB error: " + e.getMessage());
}

**Order matters in `catch` blocks** — the JVM picks the first one whose type matches. Catching `Exception` before `NumberFormatException` would never let the more specific block run, and the compiler will reject it.

Avoid catching `Exception` (the parent) just to make things compile. That swallows real bugs alongside the failure you meant to handle. Catch the specific type you're prepared to recover from; let the rest propagate.

## Throwing exceptions

Use `throw` to raise an exception explicitly. The expression after `throw` must be a `Throwable` instance — almost always one you construct with `new`.

In [ ]:
public static int parseAge(String input) {
    if (input == null) throw new NullPointerException("input was null");
    int age = Integer.parseInt(input);
    if (age < 0) throw new IllegalArgumentException("negative age: " + age);
    if (age > 150) throw new IllegalArgumentException("implausible age: " + age);
    return age;
}

The standard unchecked exceptions cover most cases. Reach for:

- **`IllegalArgumentException`** — caller passed a value outside what the method accepts
- **`IllegalStateException`** — method called when the object isn't in a state to handle it
- **`NullPointerException`** — a required reference was null (use `Objects.requireNonNull(x, "x")` to throw it with a clean message)
- **`UnsupportedOperationException`** — this operation isn't supported on this implementation

## Custom exception types

When the standard exceptions don't communicate enough, write your own. The convention is to extend `RuntimeException` for unchecked or `Exception` for checked, and provide constructors that accept a message and an optional cause.

In [ ]:
public class InsufficientFundsException extends RuntimeException {
    private final long requested;
    private final long available;

    public InsufficientFundsException(long requested, long available) {
        super("requested " + requested + " but only " + available + " available");
        this.requested = requested;
        this.available = available;
    }

    public long requested() { return requested; }
    public long available() { return available; }
}

// using it
public void withdraw(long amount) {
    if (amount > balance) {
        throw new InsufficientFundsException(amount, balance);
    }
    balance -= amount;
}

## Exception chaining — the `cause`

When you catch one exception and throw another, pass the original as the **cause**. This preserves the full chain in the stack trace and is invaluable for debugging.

In [ ]:
public Config loadConfig(Path path) {
    try {
        var json = Files.readString(path);
        return parseConfig(json);
    } catch (IOException e) {
        // wrap into a domain-specific exception, but preserve the cause
        throw new ConfigLoadException("failed to read " + path, e);
    }
}

public class ConfigLoadException extends RuntimeException {
    public ConfigLoadException(String message, Throwable cause) {
        super(message, cause);     // the Throwable(message, cause) constructor
    }
}

When you log a chained exception, the stack trace shows both layers — `ConfigLoadException: failed to read /etc/app.conf` followed by `Caused by: java.nio.file.NoSuchFileException`. You see the application-level meaning *and* the original root cause.

## `try-with-resources` — safe cleanup

Before Java 7, releasing a resource (file, stream, connection) required a `try / finally` dance with a null check. `try-with-resources` makes it one line.

Any class that implements **`AutoCloseable`** can go in the parentheses of the `try`. The JVM guarantees `.close()` is called when the block exits, exception or not. Multiple resources are allowed — they close in *reverse* declaration order.

In [ ]:
import java.nio.file.*;

// one resource
try (var lines = Files.lines(Path.of("data.txt"))) {
    lines.forEach(System.out::println);
}   // lines.close() runs automatically

// multiple resources — closed in reverse order
try (var in  = Files.newBufferedReader(Path.of("in.txt"));
     var out = Files.newBufferedWriter(Path.of("out.txt"))) {
    String line;
    while ((line = in.readLine()) != null) {
        out.write(line.toUpperCase());
        out.newLine();
    }
}   // out.close() runs first, then in.close()

You can implement `AutoCloseable` on your own classes whenever they hold something that needs cleanup — a database connection, a file handle, a network socket.

In [ ]:
public class ManagedResource implements AutoCloseable {
    public ManagedResource() { System.out.println("opened"); }
    public void use()         { System.out.println("using"); }
    @Override
    public void close()       { System.out.println("closed"); }
}

try (var r = new ManagedResource()) {
    r.use();
}
// opened
// using
// closed

## The NIO `Path` — modern filesystem paths

`java.nio.file.Path` is the modern replacement for the old `java.io.File`. A `Path` is an immutable, abstract representation of a filesystem path — it does **not** mean the path exists on disk.

In [ ]:
import java.nio.file.*;

// construction
Path p1 = Path.of("data.txt");                       // relative
Path p2 = Path.of("/Users/ganesh/Projects/java");    // absolute
Path p3 = Path.of("/Users", "ganesh", "Projects");   // joined

// queries — pure path operations, no disk access
p2.getFileName();        // "java"
p2.getParent();          // /Users/ganesh/Projects
p2.getRoot();            // /
p2.isAbsolute();         // true

// combination
Path config = p2.resolve("config.yml");              // /Users/ganesh/Projects/java/config.yml
Path rel    = p2.relativize(Path.of("/Users/ganesh/Projects/java/src"));  // src

## `Files` — reading and writing

The `Files` utility class operates on `Path` and is your one-stop shop for filesystem I/O. The high-level methods cover 90% of real use.

In [ ]:
var p = Path.of("data.txt");

// existence and metadata
Files.exists(p);
Files.isDirectory(p);
Files.size(p);             // bytes
Files.getLastModifiedTime(p);

// read a whole text file
String contents = Files.readString(p);
List<String> lines = Files.readAllLines(p);

// write a whole text file (creates or truncates)
Files.writeString(p, "hello\nworld\n");
Files.write(p, List.of("line1", "line2"));

// stream lines lazily — safe for huge files
try (var stream = Files.lines(p)) {
    long count = stream.filter(s -> !s.isBlank()).count();
}

// binary
byte[] data = Files.readAllBytes(p);
Files.write(p, data);

## Directory operations

Creating, listing, copying, deleting:

In [ ]:
var dir = Path.of("/tmp/example");

Files.createDirectories(dir);                   // mkdir -p
Files.createDirectory(dir.resolve("sub"));      // mkdir, fails if exists

// list one level — must close the stream
try (var entries = Files.list(dir)) {
    entries.forEach(System.out::println);
}

// walk recursively
try (var all = Files.walk(dir)) {
    all.filter(Files::isRegularFile)
       .forEach(System.out::println);
}

// find — walk + filter in one call
try (var matches = Files.find(dir, 5,
        (path, attrs) -> path.toString().endsWith(".java"))) {
    matches.forEach(System.out::println);
}

// copy and move
Files.copy(Path.of("a.txt"), Path.of("b.txt"), StandardCopyOption.REPLACE_EXISTING);
Files.move(Path.of("a.txt"), Path.of("c.txt"));

// delete
Files.delete(Path.of("c.txt"));            // throws if missing
Files.deleteIfExists(Path.of("c.txt"));    // safe

Every stream returned by `Files.list`, `Files.walk`, `Files.find`, and `Files.lines` holds an open file handle. **Always wrap them in `try-with-resources`** so the handle is released, even on exception.

## Serialization — a brief note

Java's built-in serialization turns an object graph into bytes you can write to a file or send over a socket, and the reverse. The mechanism is opt-in via the `Serializable` marker interface.

In [ ]:
public class Snapshot implements java.io.Serializable {
    private static final long serialVersionUID = 1L;

    private final String name;
    private final int count;
    private transient String password;    // skipped during serialization

    public Snapshot(String name, int count, String password) {
        this.name = name;
        this.count = count;
        this.password = password;
    }
}

// writing
try (var out = new ObjectOutputStream(Files.newOutputStream(Path.of("snap.bin")))) {
    out.writeObject(new Snapshot("backup", 42, "secret"));
}

// reading
try (var in = new ObjectInputStream(Files.newInputStream(Path.of("snap.bin")))) {
    Snapshot s = (Snapshot) in.readObject();
}

**A warning, not a recommendation.** Java's built-in serialization is convenient but has serious problems: it has been a recurring source of security vulnerabilities (deserializing untrusted data can execute arbitrary code), it tightly couples your class structure to its wire format, and the JDK is actively working to replace it.

For real persistence, prefer a structured format: **JSON** (Jackson, Gson), **YAML**, **Protocol Buffers**, or **Avro**. We will use Jackson extensively once we reach Spring. Treat `Serializable` as legacy knowledge — you need to recognise it in old code, but you should rarely add it to new code.

## What's next

You can now handle failure in a way callers can reason about, manage file handles safely with `try-with-resources`, and read or write any text or binary file with `Files` + `Path`. These are the boring foundations every real Java service is built on, and Spring services in particular lean on `try-with-resources` for connections and transactions all the time.

Notebook 06 turns to **concurrency**: threads, the `ExecutorService` framework that hides raw thread management, the brand-new **virtual threads** that make a million concurrent tasks cheap, `CompletableFuture` for composing asynchronous work, and a tour of the `java.util.concurrent` utilities (locks, atomics, concurrent collections).